In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from scipy.optimize import curve_fit
from scipy.special import erf

In [ ]:
fontsize = 16

# Define the four-parameter error-function model.
def error_function(x, offset, amplitude, x0, sigma):
    return offset + amplitude * 0.5 * (1 + erf((x - x0) / (np.sqrt(2) * sigma)))

# Define a Gaussian function for optional fit comparisons.
def gaussian(x, amplitude, x0, sigma):
    return amplitude * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))

In [ ]:
pwd!

In [ ]:
files = glob.glob("data/*.txt")
label_title = range(len(files))
label_index = 0
files

In [ ]:
for file_path in files:
    data = pd.read_csv(file_path)
    read_columns = [column for column in data.columns if column.startswith("read_")]

    if not read_columns:
        raise ValueError(f"No read_* columns found in {file_path}")

    x_data = pd.to_numeric(data["distance"], errors="coerce").to_numpy()
    readings = data[read_columns].apply(pd.to_numeric, errors="coerce")
    y_data = readings.mean(axis=1).to_numpy()
    y_error = readings.std(axis=1).to_numpy()

    valid = np.isfinite(x_data) & np.isfinite(y_data)
    x_data = x_data[valid]
    y_data = y_data[valid]
    y_error = y_error[valid]

    # Fit a four-parameter error function to the averaged readings.
    initial_guess = [
        y_data.min(),
        y_data.max() - y_data.min(),
        np.median(x_data),
        max((x_data.max() - x_data.min()) / 5, 1e-12),
    ]
    bounds = ([-np.inf, 0, -np.inf, 1e-12], [np.inf, np.inf, np.inf, np.inf])
    popt, _ = curve_fit(
        error_function,
        x_data,
        y_data,
        p0=initial_guess,
        bounds=bounds,
        maxfev=10000,
    )

    x_fit = np.linspace(x_data.min(), x_data.max(), 300)
    y_fit = error_function(x_fit, *popt)

    plt.figure(figsize=(10, 6))
    plt.errorbar(
        x_data,
        y_data,
        yerr=y_error,
        fmt="o",
        markersize=4,
        capsize=3,
        label="Mean measured power",
    )
    plt.plot(x_fit, y_fit, color="red", label="Error-function fit")
    plt.title(f"Knife-edge measurement: {file_path}", fontsize=fontsize)
    plt.xlabel("Distance", fontsize=fontsize)
    plt.ylabel("Power (W)", fontsize=fontsize)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    offset, amplitude, x0_fit, sigma_fit = popt
    print(
        f"{file_path}: offset={offset:.3e}, amplitude={amplitude:.3e}, "
        f"center={x0_fit:.4g}, sigma={sigma_fit:.4g}"
    )

In [ ]:
data